# SOL/USD 8h Price Prediction — Topic 10 (Mainnet) / Topic 38 (Testnet)
### Allora Forge Builder Kit — Optimized for Maximum Reward

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allora-network/allora-forge-builder-kit/blob/main/notebooks/Allora%20Forge%20Builder%20Kit.ipynb)

**So sanh voi notebook goc (example_topic_69):**

| # | Thay doi | Notebook goc | Notebook nay |
|---|----------|--------------|--------------|
| 1 | Tickers | `["btcusd"]` | `["solusd","btcusd","ethusd"]` |
| 2 | Target length | 24h | **8h** (dung topic 10/38) |
| 3 | Feature engineering | Raw OHLCV | +RSI, Bollinger, RVol, BTC corr, time |
| 4 | LightGBM params | Default | Conservative + early stopping |
| 5 | Validation | Random split | **Walk-forward** TimeSeriesSplit |
| 6 | Metrics | Pearson + DA | **7 metrics** (A+ grade system) |
| 7 | predict() | Raw features | Mirror dung feature engineering |

> **Chuan bi:** Lay API key tai [developer.allora.network](https://developer.allora.network)

## Buoc 0 — Cai dat thu vien

Chay cell nay truoc tien. Neu Runtime da co san thi bo qua.

In [ ]:
%pip install -q allora-forge-builder-kit lightgbm scikit-learn dill scipy
print("Cai dat xong")

## Buoc 1 — Cau hinh

**Chi can chinh 2 dong dau:** `FORGE_API_KEY` va `TOPIC_ID`
- `TOPIC_ID = 38` -> testnet (nen test o day truoc)
- `TOPIC_ID = 10` -> mainnet (chi dung khi model du tot)

In [ ]:
import os

FORGE_API_KEY    = "your_api_key_here"  # lay tai developer.allora.network
TOPIC_ID         = 38                   # 38=testnet SOL 8h | 10=mainnet SOL 8h
MNEMONIC         = ""                   # de trong -> tu tao vi moi, luu .allora_key

TRAIN_FROM_MONTH  = "2023-01"           # ho tro ve toi 2020
VALIDATION_MONTHS = 4
TEST_MONTHS       = 2

os.environ["ALLORA_API_KEY"] = FORGE_API_KEY
print(f"Config: Topic {TOPIC_ID} | Train tu {TRAIN_FROM_MONTH}")

## Buoc 2 — Imports

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import dill
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

from allora_forge_builder_kit import AlloraMLWorkflow, get_api_key
print("Imports OK")

## Buoc 3 — Khoi tao Workflow & Tai data

**Thay doi so voi mac dinh:**
- `tickers = ["solusd","btcusd","ethusd"]` thay vi `["btcusd"]`
- `target_length = 8` (8h ahead) thay vi 24h
- `hours_needed = 48` (lookback dai hon de tinh features)

In [ ]:
workflow = AlloraMLWorkflow(
    data_api_key           = get_api_key(),
    tickers                = ["solusd", "btcusd", "ethusd"],
    hours_needed           = 48,
    number_of_input_candles= 48,
    target_length          = 8,    # <-- 8h dung topic 10/38
)

X_train, y_train, X_val, y_val, X_test, y_test = workflow.get_train_validation_test_data(
    from_month        = TRAIN_FROM_MONTH,
    validation_months = VALIDATION_MONTHS,
    test_months       = TEST_MONTHS,
)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")
print(f"Cols  : {list(X_train.columns[:5])} ...")

## Buoc 4 — Feature Engineering

Day la noi **alpha lon nhat** duoc tao ra.

| Nhom | Features |
|------|----------|
| Momentum | RSI 6/14/24 |
| Mean reversion | Bollinger %B (3 windows) |
| Volatility | Realized Vol 8/24/48h — quan trong cho CZAR metric |
| Cross-asset | BTC log-return 1/4/8h, BTC-SOL rolling correlation & beta |
| Relative strength | SOL/BTC ratio return |
| Seasonality | sin/cos hour, sin/cos day-of-week |

> **Quan trong:** Ham `add_custom_features()` se duoc goi lai y het trong `predict()`.
> Day la diem hay bi sai nhat khi deploy (train/serve skew).

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period, min_periods=period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period, min_periods=period).mean()
    rs    = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))


def add_custom_features(df):
    df = df.copy()
    sol = df["solusd_close"]
    btc = df["btcusd_close"]
    eth = df["ethusd_close"]

    # RSI
    df["sol_rsi_6"]  = compute_rsi(sol, 6)
    df["sol_rsi_14"] = compute_rsi(sol, 14)
    df["sol_rsi_24"] = compute_rsi(sol, 24)

    # Bollinger Bands %B
    for w in [12, 20, 36]:
        mid = sol.rolling(w, min_periods=w).mean()
        std = sol.rolling(w, min_periods=w).std()
        df[f"sol_bb_pct_{w}"] = (sol - (mid - 2*std)) / (4*std + 1e-9)

    # Realized Volatility (quan trong cho CZAR metric)
    sol_lr = np.log(sol / sol.shift(1))
    df["sol_rvol_8"]  = sol_lr.rolling(8,  min_periods=4).std()
    df["sol_rvol_24"] = sol_lr.rolling(24, min_periods=12).std()
    df["sol_rvol_48"] = sol_lr.rolling(48, min_periods=24).std()

    # SOL log returns
    df["sol_ret_1h"]  = sol_lr
    df["sol_ret_4h"]  = np.log(sol / sol.shift(4))
    df["sol_ret_8h"]  = np.log(sol / sol.shift(8))
    df["sol_ret_24h"] = np.log(sol / sol.shift(24))

    # BTC cross-asset signal (alpha lon nhat)
    btc_lr = np.log(btc / btc.shift(1))
    df["btc_ret_1h"] = btc_lr
    df["btc_ret_4h"] = np.log(btc / btc.shift(4))
    df["btc_ret_8h"] = np.log(btc / btc.shift(8))

    df["sol_btc_corr_12"] = sol_lr.rolling(12, min_periods=6).corr(btc_lr)
    df["sol_btc_corr_24"] = sol_lr.rolling(24, min_periods=12).corr(btc_lr)

    cov = sol_lr.rolling(24, min_periods=12).cov(btc_lr)
    bv  = btc_lr.rolling(24, min_periods=12).var()
    df["sol_btc_beta_24"] = cov / (bv + 1e-12)

    # ETH cross-asset
    eth_lr = np.log(eth / eth.shift(1))
    df["eth_ret_1h"]      = eth_lr
    df["sol_eth_corr_24"] = sol_lr.rolling(24, min_periods=12).corr(eth_lr)

    # SOL/BTC ratio
    ratio = sol / btc
    df["sol_btc_ratio_ret8"] = np.log(ratio / ratio.shift(8))

    # Time seasonality
    if hasattr(df.index, "hour"):
        df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
    if hasattr(df.index, "dayofweek"):
        df["dow_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)

    return df


# Ap dung
X_train_fe = add_custom_features(X_train).dropna()
X_val_fe   = add_custom_features(X_val).dropna()
X_test_fe  = add_custom_features(X_test).dropna()
y_train_fe = y_train.loc[X_train_fe.index]
y_val_fe   = y_val.loc[X_val_fe.index]
y_test_fe  = y_test.loc[X_test_fe.index]

FEATURE_COLS = list(X_train_fe.columns)
print(f"Raw features tu workflow : {len(X_train.columns)}")
print(f"Custom features them vao : {len(FEATURE_COLS) - len(X_train.columns)}")
print(f"Tong features            : {len(FEATURE_COLS)}")

## Buoc 5 — Train LightGBM (Conservative Params)

| Param | Notebook goc | Notebook nay | Ly do |
|-------|-------------|--------------|-------|
| `num_leaves` | 31 | **15** | Crypto data noise cao |
| `learning_rate` | 0.1 | **0.02** | Hoc cham hon, generalize tot hon |
| `max_depth` | -1 | **5** | Tranh fit sau vao noise |
| `min_child_samples` | 20 | **50** | Moi leaf can >= 50 samples |

**Walk-forward CV** (TimeSeriesSplit) thay vi random split — tranh data leakage.

In [ ]:
LGBM_PARAMS = dict(
    objective         = "regression",
    metric            = "rmse",
    num_leaves        = 15,       # giam tu 31
    learning_rate     = 0.02,     # giam tu 0.1
    max_depth         = 5,
    min_child_samples = 50,       # tang tu 20
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.7,
    reg_alpha         = 0.1,
    reg_lambda        = 0.2,
    n_estimators      = 1000,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

# Walk-forward CV
print("Walk-forward CV (TimeSeriesSplit n=5)...")
tscv    = TimeSeriesSplit(n_splits=5)
cv_rmse = []
X_np    = X_train_fe[FEATURE_COLS].values
y_np    = y_train_fe.values

for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_np)):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(
        X_np[tr_idx], y_np[tr_idx],
        eval_set=[(X_np[vl_idx], y_np[vl_idx])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    rmse = np.sqrt(mean_squared_error(y_np[vl_idx], m.predict(X_np[vl_idx])))
    cv_rmse.append(rmse)
    print(f"  Fold {fold+1}: RMSE={rmse:.6f}  best_iter={m.best_iteration_}")

print(f"\nCV RMSE  mean={np.mean(cv_rmse):.6f}  std={np.std(cv_rmse):.6f}")

# Final model tren train+val
X_full = pd.concat([X_train_fe[FEATURE_COLS], X_val_fe[FEATURE_COLS]])
y_full = pd.concat([y_train_fe, y_val_fe])

model = lgb.LGBMRegressor(**LGBM_PARAMS)
model.fit(
    X_full, y_full,
    eval_set=[(X_val_fe[FEATURE_COLS], y_val_fe)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)],
)
print(f"\nFinal model best_iteration={model.best_iteration_}")

## Buoc 6 — Evaluation (7 Metrics Grade System)

Muc tieu: **Grade A+ (7/7)**, toi thieu **B+ (5/7)** truoc khi deploy mainnet.

In [ ]:
def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def czar_score(y_true, y_pred):
    threshold = np.percentile(np.abs(y_true), 75)
    mask = np.abs(y_true) > threshold
    if mask.sum() == 0: return 0.0
    return directional_accuracy(y_true[mask], y_pred[mask])

def wrmse_vs_zero(y_true, y_pred):
    w = np.abs(y_true) + 1e-9
    model_err = np.sqrt(np.average((y_true - y_pred)**2, weights=w))
    zero_err  = np.sqrt(np.average(y_true**2, weights=w))
    return (zero_err - model_err) / (zero_err + 1e-9)

def bootstrap_da_ci(y_true, y_pred, n=1000, ci=0.95):
    rng = np.random.default_rng(42)
    size = len(y_true)
    samples = []
    for _ in range(n):
        idx = rng.integers(0, size, size)
        samples.append(directional_accuracy(y_true[idx], y_pred[idx]))
    alpha = 1 - ci
    return np.percentile(samples, [alpha/2*100, (1-alpha/2)*100])

# Tinh metrics
y_np   = y_test_fe.values
y_pred = model.predict(X_test_fe[FEATURE_COLS])

da          = directional_accuracy(y_np, y_pred)
r_val, p_val = pearsonr(y_np, y_pred)
wrmse_imp   = wrmse_vs_zero(y_np, y_pred)
czar        = czar_score(y_np, y_pred)
da_ci       = bootstrap_da_ci(y_np, y_pred)

checks = {
    "DA >= 52%":           da >= 0.52,
    "Pearson r >= 0.05":   r_val >= 0.05,
    "Pearson p < 0.05":    p_val < 0.05,
    "WRMSE >= 5%":         wrmse_imp >= 0.05,
    "CZAR >= 10%":         czar >= 0.10,
    "DA CI lower >= 0.50": da_ci[0] >= 0.50,
    "DA CI upper >= 0.55": da_ci[1] >= 0.55,
}
passed = sum(checks.values())
grades = {7:"A+",6:"A",5:"B+",4:"B",3:"C+",2:"C",1:"D",0:"F"}
grade  = grades.get(passed, "F")
emoji  = {7:"(deploy duoc!)",6:"(tot)",5:"(du deploy testnet)",4:"(can cai thien)",3:"(chua du)",2:"(chua du)"}

print("=" * 50)
print(f"  EVALUATION — {len(y_np)} samples test")
print("=" * 50)
rows = [
    ("Directional Accuracy", f"{da:.1%}",     checks["DA >= 52%"]),
    ("Pearson r",            f"{r_val:.4f}",  checks["Pearson r >= 0.05"]),
    ("Pearson p-value",      f"{p_val:.4f}",  checks["Pearson p < 0.05"]),
    ("WRMSE improvement",    f"{wrmse_imp:.1%}", checks["WRMSE >= 5%"]),
    ("CZAR (large moves)",   f"{czar:.1%}",   checks["CZAR >= 10%"]),
    ("DA 95% CI lower",      f"{da_ci[0]:.4f}", checks["DA CI lower >= 0.50"]),
    ("DA 95% CI upper",      f"{da_ci[1]:.4f}", checks["DA CI upper >= 0.55"]),
]
for name, val, ok in rows:
    print(f"  {name:<26} {val:>8}   {'OK' if ok else '--'}")
print("=" * 50)
print(f"  GRADE: {grade}  ({passed}/7)  {emoji.get(passed,'')}")
print("=" * 50)

if passed < 5:
    print("\nKhuyen nghi de cai thien:")
    print("  * Mo rong TRAIN_FROM_MONTH ve 2022-01")
    print("  * Tang VALIDATION_MONTHS = 6")
    print("  * Them on-chain features (funding rate, open interest)")

## Buoc 7 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt

imp_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = []
for f in imp_df["feature"]:
    if "btc" in f or "eth" in f:   colors.append("#4f46e5")  # cross-asset
    elif any(x in f for x in ["rsi","bb","rvol"]): colors.append("#10b981")  # technical
    else:                           colors.append("#f59e0b")  # other

ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color=colors[::-1])
ax.set_xlabel("Importance (split count)")
ax.set_title("Top 20 Feature Importance  |  Blue=Cross-asset  Green=Technical  Yellow=Other")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

print("\nTop 10:")
for _, row in imp_df.head(10).iterrows():
    bar = chr(9608) * int(row["importance"] / imp_df["importance"].max() * 25)
    print(f"  {row['feature']:<32} {bar}")

## Buoc 8 — Export predict.pkl

> **Luu y quan trong:** `predict()` phai goi lai dung `add_custom_features()` y het luc training.
> Neu bo qua buoc nay -> live inference dung features khac training -> score kem, mat reward.

In [ ]:
def make_predict_fn(workflow, model, feature_cols, fe_fn):
    """
    Factory capture tat ca dependencies vao closure.
    Pattern nay tranh train/serve skew an toan nhat.
    """
    def predict():
        # 1. Lay raw OHLCV live tu Allora API
        live_raw = workflow.get_live_features("solusd")

        # 2. Them custom features GIONG HET luc training
        live_fe = fe_fn(live_raw).dropna()

        if live_fe.empty:
            print("WARNING: live_features trong sau dropna")
            return pd.Series(dtype=float)

        # 3. Chi dung dung feature_cols theo thu tu training
        available = [c for c in feature_cols if c in live_fe.columns]
        X_live = live_fe[available]

        preds = model.predict(X_live)
        return pd.Series(preds, index=X_live.index)

    return predict


predict = make_predict_fn(workflow, model, FEATURE_COLS, add_custom_features)

# Luu
with open("predict.pkl", "wb") as f:
    dill.dump(predict, f)
print("predict.pkl da luu")

# Verify
with open("predict.pkl", "rb") as f:
    predict_loaded = dill.load(f)
print("predict.pkl load lai thanh cong -- san sang deploy")

## Buoc 9 — Chay Worker

**Luu y:**
- Chay testnet (`TOPIC_ID=38`) truoc it nhat 24h
- Chi chuyen sang `TOPIC_ID=10` (mainnet) khi grade >= B+ (5/7)
- Worker can chay lien tuc. Google Colab co the disconnect sau ~12h.
  -> Dung Colab Pro hoac VPS de chay production.

In [ ]:
from allora_sdk.worker import AlloraWorker

with open("predict.pkl", "rb") as f:
    predict_fn = dill.load(f)

def my_model():
    tic = time.time()
    prediction = predict_fn()
    toc = time.time()
    val = float(prediction.values[-1]) if len(prediction) > 0 else 0.0
    print(f"  predict={val:.6f}  time={toc-tic:.2f}s")
    return prediction

async def main():
    worker = AlloraWorker(
        topic_id   = TOPIC_ID,
        predict_fn = my_model,
        api_key    = FORGE_API_KEY,
    )
    env = "TESTNET" if TOPIC_ID == 38 else "MAINNET"
    print(f"Worker khoi dong: Topic {TOPIC_ID} [{env}]")
    print(f"Grade: {grade}  ({passed}/7 metrics)\n")

    async for result in worker.run():
        if isinstance(result, Exception):
            print(f"  ERROR: {str(result)}")
        else:
            print(f"  SUBMITTED: {result.prediction:.6f}")

# Chay trong Colab (co san await)
await main()

## Checklist truoc khi deploy mainnet (Topic 10)

| # | Kiem tra | Buoc |
|---|----------|------|
| [ ] | Grade >= B+ (5/7 metrics) | Buoc 6 |
| [ ] | predict.pkl load lai khong loi | Buoc 8 |
| [ ] | Test testnet topic 38 >= 24h | Buoc 9 |
| [ ] | On-chain submissions duoc ghi nhan | explorer.allora.network |
| [ ] | Backup mnemonic tu `.allora_key` | Quan trong: mat la mat vi |
| [ ] | Doi TOPIC_ID = 10 | Buoc 9 |

---
**Nang cao:** Them on-chain metrics (funding rate, open interest, liquidations)
vao `add_custom_features()` de tang alpha.
SOL co tuong quan cao voi BTC va thuong lead/lag 1-2h.